# Homogeneous dataset analysis

In [ ]:

from pathlib import Path
import sys
import pickle
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "ccvnet").exists() and (candidate / "Visualization").exists():
            return candidate
    return Path(r"C:\Users\laipeiheng\Desktop\CCVNet")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ccvnet.data import DatasetConfig, parse_cell_metadata
from ccvnet.cvd import cycle_payload
from sklearn.decomposition import PCA

FIGURE_DIR = REPO_ROOT / "Visualization" / "Graph"
RESULT_CACHE_ROOT = REPO_ROOT / "results" / "homogeneous_dataset_analysis"
BATTERY_DF_PATH = RESULT_CACHE_ROOT / "battery_dataframe.csv"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

ANALYSIS_CYCLES = tuple(int(c) for c in [20, 30, 40, 50, 60, 70, 80, 90, 100])
DESCRIPTOR_FEATURES = [
    "vardQ",
    "meandQ",
    "rngdQ",
    "SOH",
    "V_at_min_dQ",
    "dQ_at_min_V",
    "centroid_V",
    "width_halfmin",
    "area_low_frac",
    "area_mid_frac",
    "area_high_frac",
]
FEATURE_ORDER = ["CVD_PC1", "CVD_PC2"] + DESCRIPTOR_FEATURES
FEATURE_LABELS = [
    r"CVD-PC$\mathsf{_1}$",
    r"CVD-PC$\mathsf{_2}$",
    "Variance",
    "Mean",
    "Range",
    "SOH",
    r"$\mathsf{F_1}$",
    r"$\mathsf{F_2}$",
    r"$\mathsf{F_3}$",
    r"$\mathsf{F_4}$",
    r"$\mathsf{F_5}$",
    r"$\mathsf{F_6}$",
    r"$\mathsf{F_7}$",
]
FEATURE_LABEL_MAP = dict(zip(FEATURE_ORDER, FEATURE_LABELS))
DATASET_ORDER = ["MICH", "XJTU", "TONGJI", "SDU", "STAN", "RWTH", "MATR", "HUST"]
AUTO_BUILD_MISSING_CACHE = False
FORCE_REBUILD_CACHE = False

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
})


def dataset_configs(base_dir: Path) -> list[DatasetConfig]:
    return [
        DatasetConfig("MICH", "MICH(NMC)", None, base_dir=base_dir, nominal_capacity_Ah=2.36, voltage_window="3.0-4.2V", default_temperature_C=25.0, default_charging_rate_C=1.0, default_discharging_rate_C=1.0),
        DatasetConfig("XJTU", "XJTU(NMC)", None, base_dir=base_dir, nominal_capacity_Ah=2.0, voltage_window="2.5-4.2V", default_temperature_C=25.0),
        DatasetConfig("TONGJI", "TONGJI(NMC)", None, base_dir=base_dir, voltage_window="2.5-4.2V"),
        DatasetConfig("SDU", "SDU(NMC)", None, base_dir=base_dir, nominal_capacity_Ah=2.4, voltage_window="2.0-4.3V", default_temperature_C=25.0),
        DatasetConfig("STAN", "STAN(NMC)", None, base_dir=base_dir, nominal_capacity_Ah=1.1, voltage_window="2.7-4.2V", default_temperature_C=30.0),
        DatasetConfig("RWTH", "RWTH(NMC)", None, base_dir=base_dir, nominal_capacity_Ah=1.1, voltage_window="3.5-3.9V", default_temperature_C=25.0, default_charging_rate_C=2.0, default_discharging_rate_C=2.0),
        DatasetConfig("MATR", "MATR(LFP)", None, base_dir=base_dir, battery_type="LFP", nominal_capacity_Ah=1.1, voltage_window="2.0-3.6V", default_temperature_C=30.0),
        DatasetConfig("HUST", "HUST(LFP)", None, base_dir=base_dir, battery_type="LFP", nominal_capacity_Ah=1.1, voltage_window="2.0-3.6V", default_temperature_C=25.0),
    ]


def _normalize_feature_name(value):
    if not isinstance(value, str):
        return value
    return value.replace("CVD_PC", "CVD_PC")


def _normalize_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [_normalize_feature_name(str(col)) for col in out.columns]
    try:
        out.index = [_normalize_feature_name(str(idx)) for idx in out.index]
    except Exception:
        pass
    for col in ["feature_name", "row_feature", "col_feature", "component"]:
        if col in out.columns:
            out[col] = out[col].map(_normalize_feature_name)
    return out


def cache_dir(name: str) -> Path:
    return RESULT_CACHE_ROOT / name


def load_cache_csv(cache_dir: Path, file_name: str, *, normalize_features: bool = False, index_col: int | None = None) -> pd.DataFrame:
    path = cache_dir / file_name
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, index_col=index_col)
    return _normalize_feature_frame(df) if normalize_features else df


def save_cache_frames(cache_name: str, frames: dict[str, pd.DataFrame]) -> None:
    target_dir = cache_dir(cache_name)
    target_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []
    for file_name, df in frames.items():
        file_path = target_dir / f"{file_name}.csv"
        if isinstance(df, pd.DataFrame):
            df.to_csv(file_path, index=False)
            n_rows = int(len(df))
        else:
            pd.DataFrame().to_csv(file_path, index=False)
            n_rows = 0
        manifest_rows.append({"frame_name": file_name, "n_rows": n_rows, "path": str(file_path)})
    pd.DataFrame(manifest_rows).to_csv(target_dir / "cache_manifest.csv", index=False)


def safe_corr(x, y):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.nanstd(x) <= 1e-12 or np.nanstd(y) <= 1e-12:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def linear_stats(x, y):
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    x = x[mask]
    y = y[mask]
    if np.nanstd(x) <= 1e-12 or np.nanstd(y) <= 1e-12:
        return np.nan, np.nan
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot <= 1e-12 else float(1.0 - ss_res / ss_tot)
    return r2, float(slope)


def feature_row_from_payload(payload: dict) -> dict:
    metrics = payload.get("metrics", {}) or {}
    shape_features = payload.get("shape_features", {}) or {}
    return {
        "vardQ": metrics.get("log10vardQ", metrics.get("vardQ", np.nan)),
        "meandQ": metrics.get("meandQ", np.nan),
        "rngdQ": metrics.get("rngdQ", np.nan),
        "SOH": payload.get("SOH", np.nan),
        "V_at_min_dQ": shape_features.get("V_at_min_dQ", np.nan),
        "dQ_at_min_V": shape_features.get("dQ_at_min_V", np.nan),
        "centroid_V": shape_features.get("centroid_V", np.nan),
        "width_halfmin": shape_features.get("width_halfmin", np.nan),
        "area_low_frac": shape_features.get("area_low_frac", np.nan),
        "area_mid_frac": shape_features.get("area_mid_frac", np.nan),
        "area_high_frac": shape_features.get("area_high_frac", np.nan),
    }


def build_pipeline_records(configs: list[DatasetConfig]) -> list[dict]:
    rows = []
    for cfg in configs:
        if not cfg.cvd_dir.exists():
            warnings.warn(f"Missing CVD directory: {cfg.cvd_dir}")
            continue
        for path in sorted(cfg.cvd_dir.glob("*CVD.pkl")):
            with path.open("rb") as handle:
                record = pickle.load(handle)
            cell = record.get("cell") or path.name.replace("_CVD.pkl", "")
            meta = parse_cell_metadata(cell, cfg)
            row = {
                "row_index": len(rows),
                "cell": cell,
                "cell_id": cell,
                "dataset_name": cfg.dataset_name,
                "target_life": float(record.get("life", np.nan)),
                "life": float(record.get("life", np.nan)),
                "battery_type": meta.get("battery_type"),
                "nominal_capacity": meta.get("nominal_capacity_Ah"),
                "operation_temperature": meta.get("operation_temperature_C"),
                "charging_rate": meta.get("charging_rate_C"),
                "discharging_rate": meta.get("discharging_rate_C"),
                "voltage_window": meta.get("voltage_window"),
                "condition_group": meta.get("condition_group", cfg.dataset_name),
                "cvd_path": str(path),
            }
            rows.append(row)
    return rows


def build_battery_dataframe(configs: list[DatasetConfig]) -> pd.DataFrame:
    selected_cycles = (20, 50, 100)
    rows = []
    for cfg in configs:
        if not cfg.cvd_dir.exists():
            continue
        for path in sorted(cfg.cvd_dir.glob("*CVD.pkl")):
            with path.open("rb") as handle:
                record = pickle.load(handle)
            cell = record.get("cell") or path.name.replace("_CVD.pkl", "")
            meta = parse_cell_metadata(cell, cfg)
            row = {
                "cell_id": cell,
                "dataset_name": cfg.dataset_name,
                "life": float(record.get("life", np.nan)),
                "battery type": meta.get("battery_type"),
                "nominal capacity": meta.get("nominal_capacity_Ah"),
                "operation temperature": meta.get("operation_temperature_C"),
                "charging rate": meta.get("charging_rate_C"),
                "discharging rate": meta.get("discharging_rate_C"),
                "voltage window": meta.get("voltage_window"),
            }
            cycle_bank = record.get("cycles", {}) or {}
            for cycle in selected_cycles:
                payload = cycle_payload(cycle_bank, int(cycle)) or {}
                metrics = payload.get("metrics", {}) or {}
                shape = payload.get("shape_features", {}) or {}
                row[f"vardQ{cycle}"] = metrics.get("log10vardQ", metrics.get("vardQ", np.nan))
                row[f"meandQ{cycle}"] = metrics.get("meandQ", np.nan)
                row[f"rngdQ{cycle}"] = metrics.get("rngdQ", np.nan)
                row[f"SOH{cycle}"] = payload.get("SOH", np.nan)
                row[f"V_at_min_dQ{cycle}"] = shape.get("V_at_min_dQ", np.nan)
                row[f"centroid_V{cycle}"] = shape.get("centroid_V", np.nan)
                row[f"width_halfmin{cycle}"] = shape.get("width_halfmin", np.nan)
                row[f"area_low_frac{cycle}"] = shape.get("area_low_frac", np.nan)
                row[f"area_mid_frac{cycle}"] = shape.get("area_mid_frac", np.nan)
                row[f"area_high_frac{cycle}"] = shape.get("area_high_frac", np.nan)
            rows.append(row)
    return pd.DataFrame(rows)


def build_cycle_feature_store(configs: list[DatasetConfig]):
    pipeline_rows = build_pipeline_records(configs)
    pipeline_df = pd.DataFrame(pipeline_rows)
    cycle_feature_store: dict[int, pd.DataFrame] = {}
    variance_rows = []
    if pipeline_df.empty:
        return pipeline_df, cycle_feature_store, pd.DataFrame()

    for cycle in ANALYSIS_CYCLES:
        curve_rows = []
        desc_rows = []
        valid_row_ids = []
        for row in pipeline_rows:
            with Path(row["cvd_path"]).open("rb") as handle:
                record = pickle.load(handle)
            payload = cycle_payload(record.get("cycles", {}) or {}, int(cycle))
            if payload is None:
                continue
            curve = np.asarray(payload.get("negative_difference", []), dtype=float)
            if curve.ndim != 1 or curve.size == 0:
                continue
            curve_rows.append(curve)
            desc_rows.append(feature_row_from_payload(payload))
            valid_row_ids.append(int(row["row_index"]))

        if not curve_rows:
            continue

        matrix = np.vstack(curve_rows).astype(float)
        finite_col_mask = np.all(np.isfinite(matrix), axis=0)
        if int(finite_col_mask.sum()) >= 3:
            matrix = matrix[:, finite_col_mask]
        else:
            matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)

        matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
        n_components = min(3, matrix.shape[0], matrix.shape[1])
        pca = PCA(n_components=n_components)
        scores = pca.fit_transform(matrix)
        explained = list(pca.explained_variance_ratio_)
        while len(explained) < 3:
            explained.append(np.nan)

        for comp_idx, ratio in enumerate(explained[:3], start=1):
            variance_rows.append({
                "cycle": int(cycle),
                "component": f"PC{comp_idx}",
                "explained_variance_ratio": ratio,
            })

        feature_df = pd.DataFrame(index=pipeline_df.index, columns=["CVD_PC1", "CVD_PC2", "CVD_PC3"] + FEATURE_ORDER[2:], dtype=float)
        for local_idx, row_idx in enumerate(valid_row_ids):
            for comp_idx in range(min(3, scores.shape[1])):
                feature_df.loc[row_idx, f"CVD_PC{comp_idx + 1}"] = float(scores[local_idx, comp_idx])
            for key, value in desc_rows[local_idx].items():
                feature_df.loc[row_idx, key] = value
        cycle_feature_store[int(cycle)] = feature_df

    variance_df = pd.DataFrame(variance_rows)
    return pipeline_df, cycle_feature_store, variance_df


def rebuild_homogeneous_caches() -> None:
    configs = dataset_configs(REPO_ROOT / "data" / "processed")
    battery_df_local = build_battery_dataframe(configs)
    if not battery_df_local.empty:
        battery_df_local.to_csv(BATTERY_DF_PATH, index=False)

    pipeline_df_local, cycle_feature_store, variance_df = build_cycle_feature_store(configs)
    if pipeline_df_local.empty or not cycle_feature_store:
        raise RuntimeError("Unable to rebuild homogeneous dataset analysis caches from local CVD data.")

    summary_df = (
        variance_df.groupby("component", dropna=False)["explained_variance_ratio"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={
            "mean": "mean_explained_variance_ratio",
            "std": "std_explained_variance_ratio",
            "count": "n_cycles",
        })
    )
    summary_df["component"] = pd.Categorical(summary_df["component"], categories=["PC1", "PC2", "PC3"], ordered=True)
    summary_df = summary_df.sort_values("component").reset_index(drop=True)
    cumulative_df = summary_df[["component", "mean_explained_variance_ratio"]].copy()
    cumulative_df["cumulative_explained_variance_ratio"] = np.cumsum(
        cumulative_df["mean_explained_variance_ratio"].fillna(0.0).to_numpy(dtype=float)
    )
    save_cache_frames("cvd_pca_variance", {
        "variance_df": variance_df,
        "summary_df": summary_df,
        "cumulative_df": cumulative_df,
    })

    cycle_corr_rows = []
    cycle_mean_corr_rows = []
    for cycle, feature_df in cycle_feature_store.items():
        sub = feature_df.reindex(columns=FEATURE_ORDER).apply(pd.to_numeric, errors="coerce")
        corr_df = sub.corr(method="pearson")
        corr_df.index.name = "row_feature"
        corr_long = corr_df.reset_index().melt(id_vars="row_feature", var_name="col_feature", value_name="pearson_r")
        corr_long["cycle"] = int(cycle)
        cycle_corr_rows.append(corr_long)
        for row_feature in FEATURE_ORDER:
            for col_feature in FEATURE_ORDER:
                cycle_mean_corr_rows.append({
                    "cycle": int(cycle),
                    "row_feature": row_feature,
                    "col_feature": col_feature,
                    "pearson_r": corr_df.loc[row_feature, col_feature] if row_feature in corr_df.index and col_feature in corr_df.columns else np.nan,
                })
    dataset_cycle_corr_df = pd.concat(cycle_corr_rows, ignore_index=True) if cycle_corr_rows else pd.DataFrame()
    mean_corr_long_df = (
        pd.DataFrame(cycle_mean_corr_rows)
        .groupby(["row_feature", "col_feature"], dropna=False)["pearson_r"]
        .mean()
        .reset_index()
        .rename(columns={"pearson_r": "mean_pearson_r"})
    )
    heatmap_df = mean_corr_long_df.pivot(index="row_feature", columns="col_feature", values="mean_pearson_r").reindex(index=FEATURE_ORDER, columns=FEATURE_ORDER)
    save_cache_frames("feature_correlation", {
        "dataset_cycle_corr_df": dataset_cycle_corr_df,
        "cycle_mean_corr_df": pd.DataFrame(cycle_mean_corr_rows),
        "mean_corr_long_df": mean_corr_long_df,
        "heatmap_df": heatmap_df,
    })

    cycle_axis = np.asarray(ANALYSIS_CYCLES, dtype=float)
    metric_rows = []
    for row_idx in pipeline_df_local.index:
        cell_name = str(pipeline_df_local.loc[row_idx, "cell"])
        dataset_name = str(pipeline_df_local.loc[row_idx, "dataset_name"])
        for feature_name in ["CVD_PC1", "CVD_PC2", "CVD_PC3"] + DESCRIPTOR_FEATURES:
            values = np.asarray([
                cycle_feature_store[cycle].loc[row_idx, feature_name] if cycle in cycle_feature_store else np.nan
                for cycle in ANALYSIS_CYCLES
            ], dtype=float)
            pearson_r = safe_corr(cycle_axis, values)
            linear_r2, slope = linear_stats(cycle_axis, values)
            metric_rows.append({
                "feature_family": "CVD" if feature_name.startswith("CVD_") else "Descriptor",
                "feature_name": feature_name,
                "row_index": int(row_idx),
                "cell": cell_name,
                "dataset_name": dataset_name,
                "pearson_r": pearson_r,
                "abs_pearson_r": np.abs(pearson_r) if pd.notna(pearson_r) else np.nan,
                "linear_r2": linear_r2,
                "slope": slope,
            })
    feature_metric_df = pd.DataFrame(metric_rows)
    feature_summary_df = (
        feature_metric_df.groupby(["feature_family", "feature_name"], dropna=False)
        .agg(
            n_cells=("row_index", "count"),
            pearson_r_mean=("pearson_r", "mean"),
            pearson_r_std=("pearson_r", "std"),
            abs_pearson_r_mean=("abs_pearson_r", "mean"),
            abs_pearson_r_std=("abs_pearson_r", "std"),
            linear_r2_mean=("linear_r2", "mean"),
            linear_r2_std=("linear_r2", "std"),
            slope_mean=("slope", "mean"),
            slope_std=("slope", "std"),
        )
        .reset_index()
    )
    save_cache_frames("cross_cycle_stability", {
        "feature_metric_df": feature_metric_df,
        "feature_summary_df": feature_summary_df,
        "cvd_pca_variance_df": variance_df,
    })

    y = pd.to_numeric(pipeline_df_local["target_life"], errors="coerce").to_numpy(dtype=float)
    cycle_corr_rows = []
    available_groups = pipeline_df_local["dataset_name"].astype(str).dropna().unique().tolist()
    group_specs = [
        (name, pipeline_df_local.index[pipeline_df_local["dataset_name"].astype(str).eq(name)].to_numpy(dtype=int))
        for name in DATASET_ORDER
        if name in available_groups
    ]
    for cycle in ANALYSIS_CYCLES:
        feature_df = cycle_feature_store.get(int(cycle))
        if feature_df is None:
            continue
        for group_name, idx in group_specs:
            y_sub = y[idx]
            feat_sub = feature_df.loc[idx, FEATURE_ORDER].copy()
            for feature_name in FEATURE_ORDER:
                cycle_corr_rows.append({
                    "group_name": group_name,
                    "cycle": int(cycle),
                    "feature_name": feature_name,
                    "pearson_r": safe_corr(feat_sub[feature_name].to_numpy(dtype=float), y_sub),
                })
    cycle_corr_df = pd.DataFrame(cycle_corr_rows)
    mean_corr_df = (
        cycle_corr_df.groupby(["group_name", "feature_name"], dropna=False)["pearson_r"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={"mean": "mean_pearson_r", "std": "std_pearson_r", "count": "n_cycles"})
    )
    featlife_heatmap_df = mean_corr_df.pivot(index="group_name", columns="feature_name", values="mean_pearson_r").reindex(index=[name for name, _ in group_specs], columns=FEATURE_ORDER)
    save_cache_frames("feature_life_correlation", {
        "cycle_corr_df": cycle_corr_df,
        "mean_corr_df": mean_corr_df,
        "heatmap_df": featlife_heatmap_df,
    })

    split_group_specs = [
        ("total split", [("Total", pipeline_df_local.index.to_numpy(dtype=int))]),
        ("per-dataset split", [(str(name), sub.index.to_numpy(dtype=int)) for name, sub in pipeline_df_local.groupby("dataset_name", sort=True)]),
        ("fine-group split", [(str(name), sub.index.to_numpy(dtype=int)) for name, sub in pipeline_df_local.groupby("condition_group", sort=True)]),
    ]
    cycle_group_rows = []
    for cycle in ANALYSIS_CYCLES:
        feature_df = cycle_feature_store.get(int(cycle))
        if feature_df is None:
            continue
        for split_name, groups in split_group_specs:
            for group_name, idx in groups:
                if len(idx) < 3:
                    continue
                y_sub = y[idx]
                feat_sub = feature_df.loc[idx, FEATURE_ORDER].copy()
                for feature_name in FEATURE_ORDER:
                    cycle_group_rows.append({
                        "split_name": split_name,
                        "group_name": group_name,
                        "cycle": int(cycle),
                        "feature_name": feature_name,
                        "pearson_r": safe_corr(feat_sub[feature_name].to_numpy(dtype=float), y_sub),
                    })
    cycle_group_corr_df = pd.DataFrame(cycle_group_rows)
    split_feature_summary_df = (
        cycle_group_corr_df.groupby(["split_name", "feature_name"], dropna=False)["pearson_r"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={"mean": "mean_cycle_corr", "std": "std_cycle_corr", "count": "n_cycles"})
    )
    split_order = ["total split", "per-dataset split", "fine-group split"]
    split_heatmap_df = split_feature_summary_df.pivot(index="split_name", columns="feature_name", values="mean_cycle_corr").reindex(index=split_order, columns=FEATURE_ORDER)
    save_cache_frames("split_feature_life_correlation", {
        "cycle_group_corr_df": cycle_group_corr_df,
        "split_feature_summary_df": split_feature_summary_df,
        "heatmap_df": split_heatmap_df,
    })


def ensure_homogeneous_analysis_cache() -> None:
    required_files = [
        cache_dir("cvd_pca_variance") / "variance_df.csv",
        cache_dir("feature_correlation") / "heatmap_df.csv",
        cache_dir("cross_cycle_stability") / "feature_summary_df.csv",
        cache_dir("feature_life_correlation") / "heatmap_df.csv",
        cache_dir("split_feature_life_correlation") / "heatmap_df.csv",
        BATTERY_DF_PATH,
    ]
    missing = [path for path in required_files if not path.exists()]
    if FORCE_REBUILD_CACHE or (AUTO_BUILD_MISSING_CACHE and missing):
        print("Rebuilding homogeneous dataset analysis caches from repo-local CVD data...")
        rebuild_homogeneous_caches()
    elif missing:
        raise FileNotFoundError(missing[0])


ensure_homogeneous_analysis_cache()

cvd_pca_cache_dir = cache_dir("cvd_pca_variance")
feature_corr_cache_dir = cache_dir("feature_correlation")
cross_cycle_cache_dir = cache_dir("cross_cycle_stability")
feature_life_cache_dir = cache_dir("feature_life_correlation")
split_feature_life_cache_dir = cache_dir("split_feature_life_correlation")

cvd_pca_variance_df = load_cache_csv(cvd_pca_cache_dir, "variance_df.csv")
cvd_pca_summary_df = load_cache_csv(cvd_pca_cache_dir, "summary_df.csv")
cvd_pca_cumulative_df = load_cache_csv(cvd_pca_cache_dir, "cumulative_df.csv")
feature_correlation_heatmap_df = load_cache_csv(feature_corr_cache_dir, "heatmap_df.csv", normalize_features=True)
if "feature_name" in feature_correlation_heatmap_df.columns:
    feature_correlation_heatmap_df = feature_correlation_heatmap_df.set_index("feature_name")
elif list(feature_correlation_heatmap_df.columns[:2]) == ["CVD_PC1", "CVD_PC2"]:
    feature_correlation_heatmap_df.index = list(feature_correlation_heatmap_df.columns)
feature_correlation_heatmap_df.index = feature_correlation_heatmap_df.index.astype(str)
feature_correlation_heatmap_df.columns = feature_correlation_heatmap_df.columns.astype(str)
feature_correlation_heatmap_df = _normalize_feature_frame(feature_correlation_heatmap_df).reindex(index=FEATURE_ORDER, columns=FEATURE_ORDER)

feature_stability_summary_df = load_cache_csv(cross_cycle_cache_dir, "feature_summary_df.csv", normalize_features=True)
feature_life_heatmap_df = load_cache_csv(feature_life_cache_dir, "heatmap_df.csv", normalize_features=True)
if "group_name" in feature_life_heatmap_df.columns:
    feature_life_heatmap_df = feature_life_heatmap_df.set_index("group_name")
elif list(feature_life_heatmap_df.columns[:2]) == ["CVD_PC1", "CVD_PC2"]:
    feature_life_heatmap_df.index = DATASET_ORDER[: len(feature_life_heatmap_df)]
feature_life_heatmap_df.index = feature_life_heatmap_df.index.astype(str)
feature_life_heatmap_df.columns = feature_life_heatmap_df.columns.astype(str)
feature_life_heatmap_df = _normalize_feature_frame(feature_life_heatmap_df).reindex(index=DATASET_ORDER, columns=FEATURE_ORDER)

split_feature_life_heatmap_df = load_cache_csv(split_feature_life_cache_dir, "heatmap_df.csv", normalize_features=True)
if "split_name" in split_feature_life_heatmap_df.columns:
    split_feature_life_heatmap_df = split_feature_life_heatmap_df.set_index("split_name")
elif list(split_feature_life_heatmap_df.columns[:2]) == ["CVD_PC1", "CVD_PC2"]:
    split_feature_life_heatmap_df.index = ["total split", "per-dataset split", "fine-group split"][: len(split_feature_life_heatmap_df)]
split_feature_life_heatmap_df.index = split_feature_life_heatmap_df.index.astype(str)
split_feature_life_heatmap_df.columns = split_feature_life_heatmap_df.columns.astype(str)
split_feature_life_heatmap_df = _normalize_feature_frame(split_feature_life_heatmap_df).reindex(index=["total split", "per-dataset split", "fine-group split"], columns=FEATURE_ORDER)

battery_summary_df = pd.read_csv(BATTERY_DF_PATH)
battery_df = battery_summary_df.rename(
    columns={
        "cell_id": "cell",
        "life": "target_life",
        "battery type": "battery_type",
        "nominal capacity": "nominal_capacity",
        "operation temperature": "operation_temperature",
        "charging rate": "charging_rate",
        "discharging rate": "discharging_rate",
        "voltage window": "voltage_window",
    }
)
battery_df["dataset_name"] = battery_df["dataset_name"].astype(str)
battery_df["target_life"] = pd.to_numeric(battery_df["target_life"], errors="coerce")
pipeline_df = battery_df.copy()
pipeline_df["row_index"] = np.arange(len(pipeline_df))
BY_GROUP_COLUMN = "dataset_name"

loaded_rows = pd.DataFrame(
    [
        {"cache": "cvd_pca_variance", "rows": len(cvd_pca_variance_df)},
        {"cache": "feature_correlation", "rows": len(feature_correlation_heatmap_df)},
        {"cache": "cross_cycle_stability", "rows": len(feature_stability_summary_df)},
        {"cache": "feature_life_correlation", "rows": len(feature_life_heatmap_df)},
        {"cache": "split_feature_life_correlation", "rows": len(split_feature_life_heatmap_df)},
        {"cache": "battery_dataframe", "rows": len(battery_df)},
    ]
)
print(f"CCVNet repo: {REPO_ROOT}")
print(f"Homogeneous analysis cache root: {RESULT_CACHE_ROOT}")
display(loaded_rows)


## CVD PCA explanatory audit

In [ ]:
component_order = ["PC1", "PC2", "PC3"]
summary_plot_df = cvd_pca_summary_df.copy()
summary_plot_df["component"] = pd.Categorical(summary_plot_df["component"], categories=component_order, ordered=True)
summary_plot_df = summary_plot_df.sort_values("component").reset_index(drop=True)
cumulative_plot_df = cvd_pca_cumulative_df.copy()
cumulative_plot_df["component"] = pd.Categorical(cumulative_plot_df["component"], categories=component_order, ordered=True)
cumulative_plot_df = cumulative_plot_df.sort_values("component").reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300)
palette = {"PC1": "#4E79A7", "PC2": "#F28E2B", "PC3": "#E15759"}
x_pos = np.arange(len(summary_plot_df))
axes[0].bar(
    x_pos,
    summary_plot_df["mean_explained_variance_ratio"].to_numpy(dtype=float),
    color=[palette[str(comp)] for comp in summary_plot_df["component"].astype(str)],
    linewidth=0.6,
    capsize=3,
    width=0.5,
)
for x, y in zip(x_pos, summary_plot_df["mean_explained_variance_ratio"].to_numpy(dtype=float)):
    axes[0].text(x, y + 0.02, f"{y:.2f}", ha="center", va="bottom", fontsize=12)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(summary_plot_df["component"].astype(str), fontsize=12)
axes[0].set_xlim(-0.6, 2.6)
axes[0].set_ylabel("Mean explained variance ratio", fontsize=12)
axes[0].set_ylim(0, 1)
axes[0].set_title("Mean variance across cycles", fontsize=12, pad=8)
axes[0].grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.35)

x_pos2 = np.arange(len(cumulative_plot_df))
axes[1].plot(
    x_pos2,
    cumulative_plot_df["cumulative_explained_variance_ratio"].to_numpy(dtype=float),
    marker="o",
    color="#2C7FB8",
    linewidth=2.0,
    markersize=5.0,
)
for xpos, value in enumerate(cumulative_plot_df["cumulative_explained_variance_ratio"].to_numpy(dtype=float)):
    axes[1].text(xpos - 0.1, value + 0.01, f"{value:.2f}", ha="center", va="bottom", fontsize=12)
axes[1].set_xticks(x_pos2)
axes[1].set_xticklabels(cumulative_plot_df["component"].astype(str), fontsize=12)
axes[1].set_xlim(-0.5, 2.5)
axes[1].set_ylim(0.8, 1.0)
axes[1].set_yticks([0.8, 0.85, 0.9, 0.95, 1.00])
axes[1].set_ylabel("Cumulative explained variance ratio", fontsize=12)
axes[1].set_title("Cumulative variance retention", fontsize=12, pad=8)
axes[1].grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.35)

fig.tight_layout()
#plt.savefig(FIGURE_DIR / "S3-2.tiff", format="tiff", dpi=500, bbox_inches="tight")
plt.show()


## Feature-feature correlation

Migrated from the source analysis workflow (`S3-1`).


In [ ]:
corr_df = feature_correlation_heatmap_df.copy()
corr_df.columns = [str(col) for col in corr_df.columns]
corr_df.index = corr_df.columns
corr_df = corr_df.reindex(index=FEATURE_ORDER, columns=FEATURE_ORDER)

values = corr_df.to_numpy(dtype=float)
abs_values = np.abs(values)
mask_upper = np.triu(np.ones_like(values, dtype=bool), k=1)
plot_values = abs_values.copy()
plot_values[mask_upper] = np.nan

fig, ax = plt.subplots(figsize=(9.2, 8.0), dpi=300)
cmap = plt.cm.GnBu.copy()
cmap.set_bad(color="white")
im = ax.imshow(plot_values, cmap=cmap, vmin=0.0, vmax=1.0)
ax.set_xticks(np.arange(len(FEATURE_LABELS)))
ax.set_xticklabels(FEATURE_LABELS, rotation=30, fontsize=12)
ax.set_yticks(np.arange(len(FEATURE_LABELS)))
ax.set_yticklabels(FEATURE_LABELS, fontsize=12)
ax.set_xticks(np.arange(-0.5, len(corr_df.columns), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(corr_df.index), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.2)
ax.tick_params(which="minor", bottom=False, left=False)

norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        if mask_upper[i, j]:
            rect = mpl.patches.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="white", edgecolor="#BDBDBD", linewidth=2.0)
            ax.add_patch(rect)
            text_val = abs_values[i, j]
            text_color = "black" if text_val < 0.5 else cmap(norm(text_val))
            ax.text(j, i, f"{text_val:.2f}", ha="center", va="center", fontsize=12, color=text_color)

for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.028, pad=0.02)
cbar.set_label("Absolute Pearson correlation", fontsize=12)
cbar.ax.tick_params(size=0)
cbar.outline.set_visible(False)

fig.tight_layout()
#plt.savefig(FIGURE_DIR / "S3-1.tiff", format="tiff", dpi=500, bbox_inches="tight")
plt.show()


## Cross-cycle feature stability

In [ ]:
plot_df = feature_stability_summary_df.copy()
plot_df = plot_df[plot_df["feature_name"].isin(FEATURE_ORDER)].copy()
plot_df["feature_name"] = pd.Categorical(plot_df["feature_name"], categories=FEATURE_ORDER, ordered=True)
plot_df = plot_df.sort_values("feature_name").reset_index(drop=True)

display_labels = [FEATURE_LABEL_MAP[str(name)] for name in plot_df["feature_name"].astype(str)]
x1 = plot_df["abs_pearson_r_mean"].to_numpy(dtype=float)
xerr1 = plot_df["abs_pearson_r_std"].fillna(0.0).to_numpy(dtype=float)
x2 = plot_df["linear_r2_mean"].to_numpy(dtype=float)
xerr2 = plot_df["linear_r2_std"].fillna(0.0).to_numpy(dtype=float)

norm = Normalize(vmin=0, vmax=1)
cmap = plt.cm.GnBu
colors1 = [cmap(norm(v)) for v in x1]
colors2 = [cmap(norm(v)) for v in x2]
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
axes[0].barh(display_labels, x1, xerr=xerr1, color=colors1, edgecolor="none", alpha=0.9, capsize=3, error_kw={"ecolor": "#444444", "linewidth": 1.5})
offset_left = max(x1) * 0.02
for i, x in enumerate(x1):
    axes[0].text(x + offset_left, i, f"{x:.2f}", va="center", ha="left", fontsize=9, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", linewidth=0))
axes[0].set_xlabel("Mean absolute Pearson correlation across cycles", fontsize=12)
axes[0].grid(axis="x", linestyle=":", linewidth=0.8, alpha=0.35)
axes[0].invert_yaxis()

axes[1].barh(display_labels, x2, xerr=xerr2, color=colors2, edgecolor="none", alpha=0.9, capsize=3, error_kw={"ecolor": "#444444", "linewidth": 1.5})
offset_right = max(x2) * 0.02
for i, x in enumerate(x2):
    axes[1].text(x + offset_right, i, f"{x:.2f}", va="center", ha="left", fontsize=9, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", linewidth=0))
axes[1].set_xlabel(r"Mean $\mathsf{R^2}$ across cycles", fontsize=11)
axes[1].grid(axis="x", linestyle=":", linewidth=0.8, alpha=0.35)
axes[1].invert_yaxis()

fig.tight_layout(rect=[0, 0, 0.91, 1])
cax = fig.add_axes([0.91, 0.135, 0.015, 0.825])
cbar = fig.colorbar(sm, cax=cax, orientation="vertical")
cbar.set_label("Value", fontsize=12)
cbar.ax.tick_params(size=0)
cbar.outline.set_visible(False)

#plt.savefig(FIGURE_DIR / "S3-3.tiff", format="tiff", dpi=500, bbox_inches="tight")
plt.show()


## Feature-life correlation across datasets


In [ ]:
heatmap_df = feature_life_heatmap_df.copy()
heatmap_df.columns = [str(col) for col in heatmap_df.columns]
heatmap_df = heatmap_df.reindex(index=[name for name in DATASET_ORDER if name in heatmap_df.index.astype(str).tolist()], columns=FEATURE_ORDER)
values = np.abs(heatmap_df.to_numpy(dtype=float))

fig, ax = plt.subplots(figsize=(10, 4), dpi=500)
im = ax.imshow(values, aspect="auto", cmap="GnBu", vmin=0.0, vmax=1.0)
ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(FEATURE_LABELS, rotation=30, fontsize=10)
ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(list(heatmap_df.index), fontsize=10)

n_rows, n_cols = values.shape
for i in range(n_rows):
    for j in range(n_cols):
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="white", linewidth=2))
        val = values[i, j]
        if np.isnan(val):
            continue
        text_color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=text_color)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)
cbar = fig.colorbar(im, ax=ax, fraction=0.028, pad=0.02)
cbar.set_label("Absolute Pearson correlation", fontsize=12)
cbar.ax.tick_params(size=0)
cbar.outline.set_visible(False)

fig.tight_layout()
#plt.savefig(FIGURE_DIR / "S3-4.tiff", format="tiff", dpi=500, bbox_inches="tight")
plt.show()


## Dataset information: life distribution and sample size

In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans']
})

dataset_palette = [
    '#a4d9e1', 
    '#91c7d7',
    '#7fb5cd',
    '#6ca1c0',
    '#598eb2',
    '#4b7ca5',
    '#3d6b98',
    '#2e5b82',
    '#1f4d6c'   
]

dataset_plot_order = ["MICH", "XJTU", "TONGJI", "SDU", "STAN", "RWTH", "MATR", "HUST"]
dataset_life_plot_df = pipeline_df.loc[:, [BY_GROUP_COLUMN, "target_life"]].copy()
dataset_life_plot_df[BY_GROUP_COLUMN] = dataset_life_plot_df[BY_GROUP_COLUMN].astype(str)
dataset_life_plot_df["target_life"] = pd.to_numeric(dataset_life_plot_df["target_life"], errors="coerce")
dataset_life_plot_df = dataset_life_plot_df.dropna(subset=["target_life"]).reset_index(drop=True)
dataset_plot_order = [name for name in dataset_plot_order if name in dataset_life_plot_df[BY_GROUP_COLUMN].unique().tolist()]

dataset_color_map = {
    name: dataset_palette[idx % len(dataset_palette)]
    for idx, name in enumerate(dataset_plot_order)
}

dataset_quantile_rows = []
for dataset_name in dataset_plot_order:
    values = dataset_life_plot_df.loc[
        dataset_life_plot_df[BY_GROUP_COLUMN].eq(dataset_name),
        "target_life",
    ].to_numpy(dtype=float)
    dataset_quantile_rows.append({
        BY_GROUP_COLUMN: dataset_name,
        "life_min": float(np.nanmin(values)),
        "life_q25": float(np.nanquantile(values, 0.25)),
        "life_median": float(np.nanmedian(values)),
        "life_q75": float(np.nanquantile(values, 0.75)),
        "life_max": float(np.nanmax(values)),
        "n_cells": int(len(values)),
    })
dataset_quantile_df = pd.DataFrame(dataset_quantile_rows)

fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
x_positions = np.arange(len(dataset_plot_order), dtype=float)

y_global_max = float(dataset_life_plot_df["target_life"].max())
for idx, dataset_name in enumerate(dataset_plot_order):
    color = dataset_color_map[dataset_name]
    values = dataset_life_plot_df.loc[
        dataset_life_plot_df[BY_GROUP_COLUMN].eq(dataset_name),
        "target_life",
    ].to_numpy(dtype=float)
    jitter = np.random.default_rng(1200 + idx).normal(0.0, 0.075, size=len(values))
    ax.scatter(
        np.full(len(values), idx, dtype=float) + jitter,
        values,
        s=20,
        color=color,
        alpha=0.4,
        edgecolor="None",
        zorder=1,
    )

    row = dataset_quantile_df.loc[dataset_quantile_df[BY_GROUP_COLUMN].eq(dataset_name)].iloc[0]
    ax.vlines(
        idx,
        float(row["life_min"]),
        float(row["life_max"]),
        color=color,
        linewidth=1.5,
        alpha=0.8,
        zorder=2,
    )
    ax.vlines(
        idx,
        float(row["life_q25"]),
        float(row["life_q75"]),
        color=color,
        linewidth=6.0,
        alpha=0.8,
        zorder=3,
    )
    ax.scatter(
        [idx],
        [float(row["life_median"])],
        s=40,
        facecolor="white",
        edgecolor=color,
        linewidth=1.5,
        zorder=4,
    )

ax.tick_params(axis="y", labelsize=10)

ax.set_xticks(x_positions)
ax.set_xticklabels(dataset_plot_order, fontsize=10)
ax.set_ylabel("Cycle life", fontsize=10)
ax.set_ylim(0, max(2800, y_global_max * 1.02))
ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.22)
ax.set_axisbelow(True)

# ========== bar plot of dataset sizes ==========
dataset_size_bar_df = (
    pipeline_df.groupby(BY_GROUP_COLUMN)["row_index"]
    .count()
    .reset_index(name="n_cells")
)
dataset_size_bar_df[BY_GROUP_COLUMN] = dataset_size_bar_df[BY_GROUP_COLUMN].astype(str)
dataset_size_bar_df = dataset_size_bar_df.loc[
    dataset_size_bar_df[BY_GROUP_COLUMN].isin(dataset_plot_order)
].copy()
dataset_size_bar_df["color"] = dataset_size_bar_df[BY_GROUP_COLUMN].map(dataset_color_map)
dataset_size_bar_df = dataset_size_bar_df.sort_values("n_cells", ascending=True).reset_index(drop=True)

ax_bar = ax.inset_axes([0.12, 0.52, 0.50, 0.40])

bar_y = np.arange(len(dataset_size_bar_df), dtype=float)
bar_values = dataset_size_bar_df["n_cells"].to_numpy(dtype=float)
bar_labels = dataset_size_bar_df[BY_GROUP_COLUMN].tolist()
bar_colors = dataset_size_bar_df["color"].tolist()

ax_bar.barh(bar_y, bar_values, color=bar_colors, alpha=0.95,
            edgecolor="white", linewidth=2, height=0.9)

ax_bar.set_xlim(0, float(bar_values.max()) * 1.50)

for yi, value in zip(bar_y, bar_values):
    ax_bar.text(value + 2.0, yi, f"{int(value)}", va="center", ha="left", fontsize=10, color="#000000")

ax_bar.set_yticks(bar_y)
ax_bar.set_yticklabels(bar_labels, fontsize=8)
ax_bar.invert_yaxis()
ax_bar.set_xticks([])
ax_bar.set_title(f"Dataset total size = {int(dataset_size_bar_df['n_cells'].sum())}", fontsize=8, pad=6, loc="left")
ax_bar.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.20)

for spine_name, spine in ax_bar.spines.items():
    if spine_name == 'left':
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_color("#333333")
    else:
        spine.set_visible(False)

ax_bar.tick_params(axis="y", length=0, pad=2)
ax_bar.set_facecolor((1, 1, 1, 0.88))

#plt.savefig(FIGURE_DIR / "F2a.tiff", dpi=500, format="tiff", bbox_inches="tight")

plt.tight_layout()
plt.show()

## Consolidated feature audit

In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans']
})

consolidated_feature_order = ["CVD_PC1", "CVD_PC2"] + list(DESCRIPTOR_FEATURES)

part1_df = feature_correlation_heatmap_df.reindex(index=consolidated_feature_order, columns=consolidated_feature_order).copy()

feature_stability_lookup = feature_stability_summary_df.set_index("feature_name")
part2_df = pd.DataFrame({
    "|r|": feature_stability_lookup.reindex(consolidated_feature_order)["abs_pearson_r_mean"],
    "R^2": feature_stability_lookup.reindex(consolidated_feature_order)["linear_r2_mean"],
}, index=consolidated_feature_order)

dataset_group_order = [
    name for name in ["MICH", "XJTU", "TONGJI", "SDU", "STAN", "RWTH", "MATR", "HUST"]
    if name in feature_life_heatmap_df.index.astype(str).tolist()
]
part3_df = feature_life_heatmap_df.T.reindex(index=consolidated_feature_order, columns=dataset_group_order).copy()

gap_col = pd.DataFrame(np.nan, index=consolidated_feature_order, columns=[""])
combined_df = pd.concat(
    [part1_df, gap_col, part2_df, gap_col.copy(), part3_df],
    axis=1,
)

combined_values = combined_df.to_numpy(dtype=float)
masked = np.ma.masked_invalid(combined_values)

part1_cols = list(part1_df.columns)
part2_cols = list(part2_df.columns)
part3_cols = list(part3_df.columns)

fig, ax = plt.subplots(figsize=(14, 4), dpi=300)

cmap = plt.colormaps["GnBu"].copy()
cmap.set_bad(color="white")

abs_values = np.abs(masked)
im = ax.imshow(abs_values, cmap=cmap, vmin=0.0, vmax=1.0, aspect="auto")

n_rows, n_cols = combined_df.shape
ax.set_xlim(-0.5, n_cols - 0.5)
ax.set_ylim(n_rows - 0.5, -0.5)

section_starts = {}
current = 0
section_starts["part1"] = current
current += len(part1_cols)
gap1_idx = current
current += 1
section_starts["part2"] = current
current += len(part2_cols)
gap2_idx = current
current += 1
section_starts["part3"] = current
current += len(part3_cols)

for gap_idx in [gap1_idx, gap2_idx]:
    ax.axvline(gap_idx, color="#6D6D6D", linewidth=2, linestyle=":", zorder=5)

for i in range(n_rows):
    for j in range(n_cols):
        if not np.isnan(combined_values[i, j]):
            rect = plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                 fill=False, edgecolor='white', linewidth=2)
            ax.add_patch(rect)

y_labels = [r"CVD-PC$\mathsf{_1}$", r"CVD-PC$\mathsf{_2}$", "Variance", "Mean", "Range", "SOH", r"$\mathsf{F_1}$", r"$\mathsf{F_2}$", r"$\mathsf{F_3}$", r"$\mathsf{F_4}$", r"$\mathsf{F_5}$", r"$\mathsf{F_6}$", r"$\mathsf{F_7}$"]
ax.set_yticks(np.arange(len(y_labels)))
ax.set_yticklabels(y_labels, fontsize=10)

x_labels = []
x_labels.extend([r"CVD-PC$\mathsf{_1}$", r"CVD-PC$\mathsf{_2}$", "Variance", "Mean", "Range", "SOH", r"$\mathsf{F_1}$", r"$\mathsf{F_2}$", r"$\mathsf{F_3}$", r"$\mathsf{F_4}$", r"$\mathsf{F_5}$", r"$\mathsf{F_6}$", r"$\mathsf{F_7}$"])
x_labels.append("")
x_labels.extend(["Corr", r"$\mathsf{R^2}$"])
x_labels.append("")
x_labels.extend(["MICH", "XJTU", "TONGJI", "SDU", "STAN", "RWTH", "MATR", "HUST"][:len(part3_cols)])

ax.set_xticks(np.arange(len(x_labels)))
ax.set_xticklabels(x_labels, fontsize=10, rotation=30, ha="center")

ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="both", length=0)

section_title_y = -1.25
ax.text(section_starts["part1"] + (len(part1_cols) - 1) / 2, section_title_y, "1. Feature-wise correlation",
        ha="center", va="center", fontsize=12, color="#000000", clip_on=False)
ax.text(section_starts["part2"] + (len(part2_cols) - 1) / 2, section_title_y, "2. Cross-cycle linearity",
        ha="center", va="center", fontsize=12, color="#000000", clip_on=False)
ax.text(section_starts["part3"] + (len(part3_cols) - 1) / 2, section_title_y, "3. Per-dataset lifetime correlation",
        ha="center", va="center", fontsize=12, color="#000000", clip_on=False)

cbar = fig.colorbar(im, ax=ax, fraction=0.022, pad=0.02)
cbar.set_label("Absolute correlation", fontsize=12)
cbar.outline.set_visible(False)
cbar.ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.subplots_adjust(left=0.16, right=0.94, top=0.86, bottom=0.23)
#plt.savefig(FIGURE_DIR / "F2c.tiff", format="tiff", dpi=500, bbox_inches="tight")
plt.show()


## Dataset information: split hierarchy table

In [ ]:
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

plt.rcParams.update({
    "text.usetex": False,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans']
})

dataset_profile_csv_path = BATTERY_DF_PATH
dataset_profile_df = pd.read_csv(dataset_profile_csv_path).copy()

dataset_profile_order = ["STAN", "TONGJI", "SDU", "RWTH", "MICH", "XJTU", "MATR", "HUST"]
dataset_profile_df["dataset_name"] = dataset_profile_df["dataset_name"].astype(str)
dataset_profile_df = dataset_profile_df.loc[dataset_profile_df["dataset_name"].isin(dataset_profile_order)].copy()
dataset_profile_order = [d for d in dataset_profile_order if d in dataset_profile_df["dataset_name"].unique().tolist()]

dataset_profile_df["battery type"] = dataset_profile_df["battery type"].fillna("unknown").astype(str)
dataset_profile_df["voltage window"] = dataset_profile_df["voltage window"].fillna("unknown").astype(str).str.replace("V", "", regex=False)
dataset_profile_df["operation temperature"] = pd.to_numeric(dataset_profile_df["operation temperature"], errors="coerce")
dataset_profile_df["nominal capacity"] = pd.to_numeric(dataset_profile_df["nominal capacity"], errors="coerce")
dataset_profile_df["charging rate"] = pd.to_numeric(dataset_profile_df["charging rate"], errors="coerce")
dataset_profile_df["discharging rate"] = pd.to_numeric(dataset_profile_df["discharging rate"], errors="coerce")


def dataset_profile_mode(series):
    s = pd.Series(series).dropna().astype(str)
    if s.empty:
        return "unknown"
    return s.value_counts().index[0]


def dataset_profile_unique_sorted(series):
    vals = pd.Series(series).dropna().tolist()
    if not vals:
        return []
    return sorted(set(vals))


def dataset_profile_fmt_capacity(sub_df):
    vals = dataset_profile_unique_sorted(pd.to_numeric(sub_df["nominal capacity"], errors="coerce").round(2))
    if not vals:
        return "unknown"
    if len(vals) == 1:
        return f"{float(vals[0]):.2f}".rstrip("0").rstrip(".")
    return "/".join(f"{float(v):.2f}".rstrip("0").rstrip(".") for v in vals)


def dataset_profile_fmt_rate(value):
    if pd.isna(value):
        return None
    return f"{float(value):.2f}".rstrip("0").rstrip(".")


def dataset_profile_protocol_label(charge_value, discharge_value):
    if pd.isna(charge_value) or pd.isna(discharge_value):
        return "unknown"
    return f"{dataset_profile_fmt_rate(charge_value)}/{dataset_profile_fmt_rate(discharge_value)}"


dataset_profile_counts = (
    dataset_profile_df.groupby("dataset_name").size().reindex(dataset_profile_order).fillna(0).astype(int)
)
dataset_profile_total = int(dataset_profile_counts.sum())
dataset_profile_widths = (dataset_profile_counts / dataset_profile_total).to_dict()

dataset_profile_rows = []
for dataset_name in dataset_profile_order:
    sub = dataset_profile_df.loc[dataset_profile_df["dataset_name"].eq(dataset_name)].copy()
    chem = dataset_profile_mode(sub["battery type"]).replace("/carbon", "")
    voltage_vals = dataset_profile_unique_sorted(sub["voltage window"])
    voltage = voltage_vals[0] if len(voltage_vals) == 1 else "various"
    capacity = dataset_profile_fmt_capacity(sub)
    dataset_profile_rows.append({
        "dataset_name": dataset_name,
        "n_cells": int(len(sub)),
        "chemistry": chem,
        "voltage": voltage,
        "capacity": capacity,
    })
dataset_profile_summary_df = pd.DataFrame(dataset_profile_rows).set_index("dataset_name")

dataset_profile_edges = {}
cursor = 0.0
for dataset_name in dataset_profile_order:
    width = float(dataset_profile_widths[dataset_name])
    dataset_profile_edges[dataset_name] = (cursor, cursor + width)
    cursor += width

# ============================================================
# RIGHT COLUMN VALUES - Manually define here
# Order must match dataset_profile_row_names below
# ============================================================
TEMP_ROW_LABEL = r"Temperature (°C)"

RIGHT_COLUMN_VALUES = [
    "Total",    # Total dataset row
    "8",        # Dataset row
    "2",        # Chemistry row
    "4",        # Temperature (°C) row
    "7",        # Voltage window (V) row
    "173",      # Cycling protocol row
    "7",        # Nominal capacity (Ah) row
]

# ============================================================
# FONT SIZE CONTROL
# ============================================================
FONTSIZE_LEFT = 12
FONTSIZE_MIDDLE = 10
FONTSIZE_RIGHT = 12

dataset_profile_row_names = [
    "Total dataset",
    "Dataset",
    "Chemistry",
    TEMP_ROW_LABEL,
    "Voltage window (V)",
    "Cycling protocol",
    "Nominal capacity (Ah)",
]
dataset_profile_row_y = {name: 6 - idx for idx, name in enumerate(dataset_profile_row_names)}

fig, ax = plt.subplots(figsize=(13.0, 4.9), dpi=300)
ax.set_xlim(-0.21, 1.28)
ax.set_ylim(0.0, 7.0)
ax.axis("off")

left_x0, left_x1 = -0.21, -0.005
main_x0, main_x1 = 0.0, 1.0
right_x0, right_x1 = 1.01, 1.10
row_h = 0.9
row_gap = 0.05

face_a = "#c9dbee"
face_b = "#c3cad3"
edge_c_middle = "white"
edge_c_side = "#A8A8A8"
label_face = "white"
count_face = "white"

# ============================================================
# DYNAMIC COLOR SYSTEM - Blue gradient (Temperature, Protocol, Capacity)
# ============================================================
c_min_blue = np.array(mpl.colors.to_rgb("#c9dbee"))
c_max_blue = np.array(mpl.colors.to_rgb("#116FAD"))
MAX_BOXES = 172


def get_split_colors(n_boxes, max_boxes=MAX_BOXES, c_min=c_min_blue, c_max=c_max_blue):
    if n_boxes <= 0:
        return []
    half_max = max_boxes / 2
    scale = min(1.0, n_boxes / half_max)
    scale = np.sqrt(scale)
    n_samples = max(n_boxes, 2)
    positions = np.linspace(0, scale, n_samples)
    colors = []
    for pos in positions:
        color = c_min * (1.0 - pos) + c_max * pos
        colors.append(color)
    return colors


# ============================================================
# DYNAMIC COLOR SYSTEM - Gray gradient (Voltage, Capacity)
# ============================================================
c_min_gray = np.array(mpl.colors.to_rgb("#d7dde3"))
c_max_gray = np.array(mpl.colors.to_rgb("#8f98a3"))


def draw_profile_box(x0, x1, y0, text, facecolor, edgecolor, fontsize=12, lw=0.9, fontweight="normal"):
    """Draw a single box with text, supporting font weight."""
    rect = patches.Rectangle(
        (x0, y0),
        x1 - x0,
        row_h,
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=lw,
    )
    ax.add_patch(rect)
    if text:
        ax.text(
            (x0 + x1) / 2,
            y0 + row_h / 2,
            text,
            ha="center",
            va="center",
            fontsize=fontsize,
            color="#000000",
            fontweight=fontweight,
        )


def draw_split_row_with_dynamic_colors(row_name, dataset_to_segments, right_value, 
                                        add_borders=False, c_min=c_min_blue, c_max=c_max_blue):
    """
    Draw a split row where each dataset's segments use colors from the dynamic gradient.
    """
    y0 = dataset_profile_row_y[row_name] + row_gap / 2
    
    for dataset_name in dataset_profile_order:
        x0, x1 = dataset_profile_edges[dataset_name]
        segments = dataset_to_segments.get(dataset_name, [])
        
        if not segments:
            draw_profile_box(x0, x1, y0, "", face_a, edge_c_middle, fontsize=FONTSIZE_MIDDLE)
            continue
        
        total = max(1, sum(seg["count"] for seg in segments))
        n_seg = len(segments)
        colors = get_split_colors(n_seg, c_min=c_min, c_max=c_max)
        
        cursor = x0
        for idx, seg in enumerate(segments):
            width = (x1 - x0) * float(seg["count"]) / total
            color = colors[idx % len(colors)] if colors else c_min
            
            rect = patches.Rectangle(
                (cursor, y0),
                width,
                row_h,
                facecolor=color,
                edgecolor=edge_c_middle if add_borders else "none",
                linewidth=0.9 if add_borders else 0,
            )
            ax.add_patch(rect)
            
            if width > 0.032 and seg["label"]:
                text_color = "white" if np.mean(color) < 0.5 else "#000000"
                ax.text(
                    cursor + width / 2,
                    y0 + row_h / 2,
                    seg["label"],
                    ha="center",
                    va="center",
                    fontsize=FONTSIZE_MIDDLE,
                    color=text_color,
                )
            cursor += width
        
        ax.add_patch(
            patches.Rectangle(
                (x0, y0),
                x1 - x0,
                row_h,
                fill=False,
                edgecolor=edge_c_middle,
                linewidth=0.9,
            )
        )
    
    draw_profile_box(right_x0, right_x1, y0, right_value, count_face, edge_c_side, fontsize=FONTSIZE_RIGHT, fontweight="bold")


def draw_merged_profile_row(field_name, row_name, right_value, facecolor):
    """Draw a row where adjacent boxes with identical values are merged."""
    y0 = dataset_profile_row_y[row_name] + row_gap / 2
    start_dataset = dataset_profile_order[0]
    current_label = str(dataset_profile_summary_df.loc[start_dataset, field_name])
    start_x = dataset_profile_edges[start_dataset][0]
    prev_dataset = start_dataset
    for dataset_name in dataset_profile_order[1:]:
        label = str(dataset_profile_summary_df.loc[dataset_name, field_name])
        if label != current_label:
            x1 = dataset_profile_edges[prev_dataset][1]
            draw_profile_box(start_x, x1, y0, current_label, facecolor, edge_c_middle, fontsize=FONTSIZE_MIDDLE)
            start_x = dataset_profile_edges[dataset_name][0]
            current_label = label
        prev_dataset = dataset_name
    x1 = dataset_profile_edges[prev_dataset][1]
    draw_profile_box(start_x, x1, y0, current_label, facecolor, edge_c_middle, fontsize=FONTSIZE_MIDDLE)
    draw_profile_box(right_x0, right_x1, y0, right_value, count_face, edge_c_side, fontsize=FONTSIZE_RIGHT, fontweight="bold")


# --- Left column ---
for row_name in dataset_profile_row_names:
    y0 = dataset_profile_row_y[row_name] + row_gap / 2
    draw_profile_box(left_x0, left_x1, y0, row_name, label_face, edge_c_side, fontsize=FONTSIZE_LEFT)

# --- Total dataset row ---
y0 = dataset_profile_row_y["Total dataset"] + row_gap / 2
draw_profile_box(main_x0, main_x1, y0, "Total dataset", face_b, edge_c_middle, fontsize=FONTSIZE_MIDDLE)
draw_profile_box(right_x0, right_x1, y0, RIGHT_COLUMN_VALUES[0], count_face, edge_c_side, fontsize=FONTSIZE_RIGHT, fontweight="bold")

# --- Dataset row ---
y0 = dataset_profile_row_y["Dataset"] + row_gap / 2
for dataset_name in dataset_profile_order:
    x0, x1 = dataset_profile_edges[dataset_name]
    draw_profile_box(x0, x1, y0, dataset_name, face_a, edge_c_middle, fontsize=FONTSIZE_MIDDLE)
draw_profile_box(right_x0, right_x1, y0, RIGHT_COLUMN_VALUES[1], count_face, edge_c_side, fontsize=FONTSIZE_RIGHT, fontweight="bold")

# --- Chemistry row ---
draw_merged_profile_row("chemistry", "Chemistry", RIGHT_COLUMN_VALUES[2], face_b)

# --- Temperature row (blue gradient) ---
temperature_segments = {}
for dataset_name in dataset_profile_order:
    sub = dataset_profile_df.loc[dataset_profile_df["dataset_name"].eq(dataset_name)].copy()
    counts = (
        sub["operation temperature"]
        .dropna()
        .round(0)
        .astype(int)
        .value_counts()
        .sort_index()
    )
    if counts.empty:
        temperature_segments[dataset_name] = [{"label": "", "count": int(len(sub))}]
    else:
        temperature_segments[dataset_name] = [
            {"label": str(int(temp)), "count": int(count)}
            for temp, count in counts.items()
        ]
draw_split_row_with_dynamic_colors(TEMP_ROW_LABEL, temperature_segments, RIGHT_COLUMN_VALUES[3], add_borders=False)

# --- Voltage window row (gray gradient) ---
voltage_segments = {}
for dataset_name in dataset_profile_order:
    sub = dataset_profile_df.loc[dataset_profile_df["dataset_name"].eq(dataset_name)].copy()
    counts = sub["voltage window"].astype(str).value_counts()
    if len(counts) <= 1:
        label = counts.index[0] if len(counts) == 1 else str(dataset_profile_summary_df.loc[dataset_name, "voltage"])
        voltage_segments[dataset_name] = [{"label": str(label), "count": int(len(sub))}]
    else:
        ordered = counts.sort_values(ascending=False)
        voltage_segments[dataset_name] = [
            {"label": str(label), "count": int(count)}
            for label, count in ordered.items()
        ]
draw_split_row_with_dynamic_colors("Voltage window (V)", voltage_segments, RIGHT_COLUMN_VALUES[4], 
                                    add_borders=False, c_min=c_min_gray, c_max=c_max_gray)

# --- Nominal capacity row (gray gradient, split only where needed, e.g. TONGJI) ---
capacity_segments = {}
for dataset_name in dataset_profile_order:
    sub = dataset_profile_df.loc[dataset_profile_df["dataset_name"].eq(dataset_name)].copy()
    counts = (
        pd.to_numeric(sub["nominal capacity"], errors="coerce")
        .round(2)
        .value_counts()
        .sort_index()
    )
    if counts.empty:
        capacity_segments[dataset_name] = [{"label": "", "count": int(len(sub))}]
    elif len(counts) == 1:
        value = list(counts.index)[0]
        capacity_segments[dataset_name] = [{"label": f"{float(value):.2f}".rstrip("0").rstrip("."), "count": int(len(sub))}]
    else:
        capacity_segments[dataset_name] = [
            {"label": f"{float(cap):.2f}".rstrip("0").rstrip("."), "count": int(count)}
            for cap, count in counts.items()
        ]
draw_split_row_with_dynamic_colors("Nominal capacity (Ah)", capacity_segments, RIGHT_COLUMN_VALUES[6], 
                                    add_borders=False, c_min=c_min_gray, c_max=c_max_gray)

# ============================================================
# CYCLING PROTOCOL ROW (blue gradient)
# MATR and HUST: individual cell coloring (no borders)
# Others: grouped by protocol label
# ============================================================
proto_y0 = dataset_profile_row_y["Cycling protocol"] + row_gap / 2

for dataset_name in dataset_profile_order:
    x0, x1 = dataset_profile_edges[dataset_name]
    sub = dataset_profile_df.loc[dataset_profile_df["dataset_name"].eq(dataset_name)].copy()
    
    proto_series = pd.Series(
        [dataset_profile_protocol_label(c, d) for c, d in zip(sub["charging rate"], sub["discharging rate"])],
        index=sub.index,
        dtype=object,
    )

    if dataset_name in ["MATR", "HUST"]:
        n_boxes = len(sub)
        if n_boxes > 0:
            colors = get_split_colors(n_boxes, c_min=c_min_blue, c_max=c_max_blue)
            edges = [x0 + (x1 - x0) * k / n_boxes for k in range(n_boxes + 1)]
            
            for k in range(n_boxes):
                color = colors[k % len(colors)] if colors else c_min_blue
                rect = patches.Rectangle(
                    (edges[k], proto_y0),
                    edges[k + 1] - edges[k],
                    row_h,
                    facecolor=color,
                    edgecolor="none",
                    linewidth=0,
                )
                ax.add_patch(rect)
                
                label = str(proto_series.iloc[k])
                width = edges[k + 1] - edges[k]
                if width > 0.025 and label != "unknown":
                    text_color = "white" if np.mean(color) < 0.5 else "#000000"
                    ax.text(
                        edges[k] + width / 2,
                        proto_y0 + row_h / 2,
                        label,
                        ha="center",
                        va="center",
                        fontsize=FONTSIZE_MIDDLE - 1,
                        color=text_color,
                    )
        else:
            draw_profile_box(x0, x1, proto_y0, "", face_a, edge_c_middle, fontsize=FONTSIZE_MIDDLE)
    
    else:
        counts = proto_series.value_counts(dropna=False)
        n_boxes = len(counts)
        total = max(1, int(counts.sum()))
        colors = get_split_colors(n_boxes, c_min=c_min_blue, c_max=c_max_blue)
        
        cursor = x0
        for idx, (label, count) in enumerate(counts.items()):
            width = (x1 - x0) * float(count) / total
            color = colors[idx % len(colors)] if colors else c_min_blue
            
            rect = patches.Rectangle(
                (cursor, proto_y0),
                width,
                row_h,
                facecolor=color,
                edgecolor="none",
                linewidth=0,
            )
            ax.add_patch(rect)
            
            if width > 0.032 and label != "unknown":
                text_color = "white" if np.mean(color) < 0.5 else "#000000"
                ax.text(
                    cursor + width / 2,
                    proto_y0 + row_h / 2,
                    str(label),
                    ha="center",
                    va="center",
                    fontsize=FONTSIZE_MIDDLE,
                    color=text_color,
                )
            cursor += width
    
    ax.add_patch(
        patches.Rectangle((x0, proto_y0), x1 - x0, row_h, fill=False, edgecolor=edge_c_middle, linewidth=1)
    )

draw_profile_box(right_x0, right_x1, proto_y0, RIGHT_COLUMN_VALUES[5], count_face, edge_c_side, fontsize=FONTSIZE_RIGHT, fontweight="bold")

# ============================================================
# BRACKETS
# ============================================================
brace_x = 1.12
text_x = 1.13


def draw_profile_bracket(y_bottom, y_top, text):
    ax.add_line(Line2D([brace_x, brace_x], [y_bottom, y_top], color="#000000", linewidth=1.2))
    ax.add_line(Line2D([brace_x - 0.01, brace_x], [y_bottom, y_bottom], color="#000000", linewidth=1.2))
    ax.add_line(Line2D([brace_x - 0.01, brace_x], [y_top, y_top], color="#000000", linewidth=1.2))
    ax.text(text_x, (y_bottom + y_top) / 2, text, ha="left", va="center", fontsize=12, color="#000000")


row_bottoms = {name: dataset_profile_row_y[name] + row_gap / 2 for name in dataset_profile_row_names}
row_tops = {name: dataset_profile_row_y[name] + row_gap / 2 + row_h for name in dataset_profile_row_names}

draw_profile_bracket(row_bottoms["Total dataset"], row_tops["Total dataset"], "Total split")
draw_profile_bracket(row_bottoms["Dataset"], row_tops["Dataset"], "Per-dataset split")
draw_profile_bracket(row_bottoms["Voltage window (V)"], row_tops["Chemistry"], "Fine-group split")

plt.tight_layout()
#plt.savefig(FIGURE_DIR / "F2b.tiff", dpi=500, format="tiff", bbox_inches="tight")
plt.show()
